In [1]:
import numpy as np
import joblib
import os
import pandas as pd
from statsmodels.stats.multitest import multipletests

tte_threshold = 20


suffixes = []
hospital_types = ['high_resource']
strategies = ['covariate']
thresholds = [30, 40, 60, 80]
models = ['icind', 'pcind', 'dhcind', 'proposed_smallweight_linear', 'proposed_smallweight_exp', 'proposed_largeweight_linear', 'proposed_largeweight_exp']
model_names = ['IPCW', "Standard\nMargin", 'Standard', 'Proposed - Linear, Small Weight', 'Proposed - Exponential, Small Weight', 'Proposed - Linear, Large Weight', 'Proposed - Exponential, Large Weight']

result_dir = '$CWITE_DATA_ROOT/support50_propbin_data'
preds_dir = '$CWITE_DATA_ROOT/support50_propbin_test_preds'

# Build suffixes
for strategy in strategies:
    if strategy == 'hybrid':
        suffixes.extend([f'{strategy}_T{t}_{h}' for t in thresholds for h in hospital_types])
    elif strategy == 'outcome':
        suffixes.extend([f'{strategy}_T{t}' for t in thresholds])
    elif strategy == 'covariate':
        suffixes.extend([f'{strategy}_{h}' for h in hospital_types])

# Iterate over each suffix and group
for suffix in suffixes:
    orig_y_test = joblib.load(os.path.join(result_dir, f'orig_y_test_{suffix}.joblib'))
    print(np.median(orig_y_test))
    binary_y_test = joblib.load(os.path.join(result_dir, f'binary_y_test_{suffix}.joblib'))

    masks = {
        'Censored': binary_y_test == 0,
        'Uncensored': binary_y_test == 1
    }

    print(f"==== {suffix} ====")
    for group_short, mask in masks.items():
        if not np.any(mask):
            print(f"[{suffix}] No {group_short} samples — skipping.")
            continue

        mae_vals_by_model = {}
        for model in models:
            preds = []
            for splitidx in range(1000):
                pred_path = os.path.join(preds_dir, f'{model}_y_test_pred_split{splitidx}_{suffix}.joblib')
                preds.append(joblib.load(pred_path))
            preds = np.array(preds)

            idx = np.where(mask)[0]
            mae_vals = np.mean(np.abs(preds[:, idx] * 40.76 - orig_y_test[idx] * 40.76), axis=1)
            mae_vals_by_model[model] = mae_vals

        # Print raw MAE results
        print(f"{group_short}")
        for model, name in zip(models, model_names):
            mae = np.median(mae_vals_by_model[model])
            lo = np.percentile(mae_vals_by_model[model], 2.5)
            hi = np.percentile(mae_vals_by_model[model], 97.5)
            print(f"{name}, {mae:.2f}, 95\\% CI: [{lo:.2f}, {hi:.2f}]")

        raw_pvals = []
        compared_models = []

        for model in models:
            if model == 'proposed_smallweight_linear':
                continue
            diffs = mae_vals_by_model['proposed_smallweight_linear'] - mae_vals_by_model[model]
            pval = np.mean(diffs < 0)  # Proposed MAE < Other MAE
            raw_pvals.append(pval)
            compared_models.append(model)

        # BH-FDR correction
        reject, pvals_corrected, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')

        print("\nOne-sided p-values (Proposed Linear MAE < Other MAE):")
        for model, p_raw, p_corr, rej in zip(compared_models, raw_pvals, pvals_corrected, reject):
            method_name = model_names[models.index(model)]
            star = "*" if rej else ""
            print(f"Proposed < {method_name}: p = {p_raw:.3f}, BH-adjusted p = {p_corr:.3f} {star}")
        print()


        raw_pvals = []
        compared_models = []

        for model in models:
            if model == 'proposed_smallweight_exp':
                continue
            diffs = mae_vals_by_model['proposed_smallweight_exp'] - mae_vals_by_model[model]
            pval = np.mean(diffs < 0)  # Proposed MAE < Other MAE
            raw_pvals.append(pval)
            compared_models.append(model)

        # BH-FDR correction
        reject, pvals_corrected, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')

        print("\nOne-sided p-values (Proposed Exp MAE < Other MAE):")
        for model, p_raw, p_corr, rej in zip(compared_models, raw_pvals, pvals_corrected, reject):
            method_name = model_names[models.index(model)]
            star = "*" if rej else ""
            print(f"Proposed < {method_name}: p = {p_raw:.3f}, BH-adjusted p = {p_corr:.3f} {star}")
        print()

20.0
==== covariate_high_resource ====
Censored
IPCW, 518.22, 95\% CI: [476.45, 576.18]
Standard
Margin, 675.27, 95\% CI: [527.51, 926.30]
Standard, 876.98, 95\% CI: [811.91, 943.05]
Proposed - Linear, Small Weight, 451.71, 95\% CI: [433.15, 472.22]
Proposed - Exponential, Small Weight, 483.72, 95\% CI: [428.91, 563.39]
Proposed - Linear, Large Weight, 457.41, 95\% CI: [439.82, 476.67]
Proposed - Exponential, Large Weight, 528.66, 95\% CI: [467.34, 613.69]

One-sided p-values (Proposed Linear MAE < Other MAE):
Proposed < IPCW: p = 1.000, BH-adjusted p = 1.000 
Proposed < Standard
Margin: p = 0.991, BH-adjusted p = 1.000 
Proposed < Standard: p = 1.000, BH-adjusted p = 1.000 
Proposed < Proposed - Exponential, Small Weight: p = 0.835, BH-adjusted p = 1.000 
Proposed < Proposed - Linear, Large Weight: p = 0.889, BH-adjusted p = 1.000 
Proposed < Proposed - Exponential, Large Weight: p = 0.993, BH-adjusted p = 1.000 


One-sided p-values (Proposed Exp MAE < Other MAE):
Proposed < IPCW: p 

In [2]:
# Define TTE threshold in normalized units
for suffix in suffixes:
    orig_y_test = joblib.load(os.path.join(result_dir, f'orig_y_test_{suffix}.joblib'))
    binary_y_test = joblib.load(os.path.join(result_dir, f'binary_y_test_{suffix}.joblib'))

    masks = {
        'Censored': binary_y_test == 0,
        'Uncensored': binary_y_test == 1
    }

    print(f"==== {suffix} ====")
    for group_short, mask in masks.items():
        if not np.any(mask):
            print(f"[{suffix}] No {group_short} samples — skipping.")
            continue

        # 🔥 Split by TTE <range and ≥range
        tte_splits = {
            '<range days': mask & (orig_y_test < tte_threshold),
            '≥range days': mask & (orig_y_test >= tte_threshold)
        }


        for tte_label, tte_mask in tte_splits.items():
            print(tte_label, group_short, sum(tte_mask))


==== covariate_high_resource ====
<range days Censored 581
≥range days Censored 574
<range days Uncensored 31
≥range days Uncensored 55


In [3]:
# Define TTE threshold in normalized units
for suffix in suffixes:
    orig_y_test = joblib.load(os.path.join(result_dir, f'orig_y_test_{suffix}.joblib'))
    binary_y_test = joblib.load(os.path.join(result_dir, f'binary_y_test_{suffix}.joblib'))

    masks = {
        'Censored': binary_y_test == 0,
        'Uncensored': binary_y_test == 1
    }

    print(f"==== {suffix} ====")
    for group_short, mask in masks.items():
        if not np.any(mask):
            print(f"[{suffix}] No {group_short} samples — skipping.")
            continue

        # 🔥 Split by TTE <range and ≥range
        tte_splits = {
            '<range days': mask & (orig_y_test < tte_threshold),
            '≥range days': mask & (orig_y_test >= tte_threshold)
        }


        for tte_label, tte_mask in tte_splits.items():
            print(tte_label, group_short, sum(tte_mask))
            if not np.any(tte_mask):
                print(f"[{suffix}] No {group_short} samples in TTE {tte_label} — skipping.")
                continue

            print(f"{group_short} — TTE {tte_label}")
            mae_vals_by_model = {}
            for model in models:
                preds = []
                for splitidx in range(1000):
                    pred_path = os.path.join(preds_dir, f'{model}_y_test_pred_split{splitidx}_{suffix}.joblib')
                    preds.append(joblib.load(pred_path))
                preds = np.array(preds)

                idx = np.where(tte_mask)[0]
                mae_vals = np.mean(np.abs(preds[:, idx] * 40.76 - orig_y_test[idx] * 40.76), axis=1)
                mae_vals_by_model[model] = mae_vals

            # Print raw MAE results
            for model, name in zip(models, model_names):
                mae = np.median(mae_vals_by_model[model])
                lo = np.percentile(mae_vals_by_model[model], 2.5)
                hi = np.percentile(mae_vals_by_model[model], 97.5)
                print(f"{name}, {mae:.2f}, 95\\% CI: [{lo:.2f}, {hi:.2f}]")

            # Resampling test: Proposed better than each other model
            raw_pvals = []
            compared_models = []
    
            for model in models:
                if model == 'proposed_smallweight_linear':
                    continue
                diffs = mae_vals_by_model[model] - mae_vals_by_model['proposed_smallweight_linear']
                pval = np.mean(diffs < 0)  # Proposed MAE < Other MAE
                raw_pvals.append(pval)
                compared_models.append(model)
    
            # BH-FDR correction
            reject, pvals_corrected, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')
    
            print("\nOne-sided p-values (Proposed Linear MAE < Other MAE):")
            for model, p_raw, p_corr, rej in zip(compared_models, raw_pvals, pvals_corrected, reject):
                method_name = model_names[models.index(model)]
                star = "*" if rej else ""
                print(f"Proposed < {method_name}: p = {p_raw:.3f}, BH-adjusted p = {p_corr:.3f} {star}")
            print()
    
    
            raw_pvals = []
            compared_models = []
    
            for model in models:
                if model == 'proposed_smallweight_exp':
                    continue
                diffs = mae_vals_by_model[model] - mae_vals_by_model['proposed_smallweight_exp']
                pval = np.mean(diffs < 0)  # Proposed MAE < Other MAE
                raw_pvals.append(pval)
                compared_models.append(model)
    
            # BH-FDR correction
            reject, pvals_corrected, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')
    
            print("\nOne-sided p-values (Proposed Exp MAE < Other MAE):")
            for model, p_raw, p_corr, rej in zip(compared_models, raw_pvals, pvals_corrected, reject):
                method_name = model_names[models.index(model)]
                star = "*" if rej else ""
                print(f"Proposed < {method_name}: p = {p_raw:.3f}, BH-adjusted p = {p_corr:.3f} {star}")
            print()

==== covariate_high_resource ====
<range days Censored 581
Censored — TTE <range days
IPCW, 639.88, 95\% CI: [543.41, 760.28]
Standard
Margin, 834.14, 95\% CI: [300.37, 1329.83]
Standard, 1328.66, 95\% CI: [1224.20, 1427.31]
Proposed - Linear, Small Weight, 368.52, 95\% CI: [338.49, 399.05]
Proposed - Exponential, Small Weight, 591.72, 95\% CI: [420.28, 795.09]
Proposed - Linear, Large Weight, 366.31, 95\% CI: [338.85, 395.05]
Proposed - Exponential, Large Weight, 667.63, 95\% CI: [484.32, 875.48]

One-sided p-values (Proposed Linear MAE < Other MAE):
Proposed < IPCW: p = 0.000, BH-adjusted p = 0.000 *
Proposed < Standard
Margin: p = 0.062, BH-adjusted p = 0.074 
Proposed < Standard: p = 0.000, BH-adjusted p = 0.000 *
Proposed < Proposed - Exponential, Small Weight: p = 0.004, BH-adjusted p = 0.006 *
Proposed < Proposed - Linear, Large Weight: p = 0.630, BH-adjusted p = 0.630 
Proposed < Proposed - Exponential, Large Weight: p = 0.000, BH-adjusted p = 0.000 *


One-sided p-values (Prop

In [4]:
# Define TTE threshold in normalized units

for suffix in suffixes:
    orig_y_test = joblib.load(os.path.join(result_dir, f'orig_y_test_{suffix}.joblib'))
    binary_y_test = joblib.load(os.path.join(result_dir, f'binary_y_test_{suffix}.joblib'))

    masks = {
        'Censored': binary_y_test == 0,
        'Uncensored': binary_y_test == 1
    }

    print(f"==== {suffix} ====")
    for group_short, mask in masks.items():
        if not np.any(mask):
            print(f"[{suffix}] No {group_short} samples — skipping.")
            continue

        # 🔥 Split by TTE <range and ≥range
        tte_splits = {
            '<range days': mask & (orig_y_test < tte_threshold),
            '≥range days': mask & (orig_y_test >= tte_threshold)
        }

        for tte_label, tte_mask in tte_splits.items():
            if not np.any(tte_mask):
                print(f"[{suffix}] No {group_short} samples in TTE {tte_label} — skipping.")
                continue

            print(f"{group_short} — TTE {tte_label}")
            mae_vals_by_model = {}
            for model in models:
                preds = []
                for splitidx in range(1000):
                    pred_path = os.path.join(preds_dir, f'{model}_y_test_pred_split{splitidx}_{suffix}.joblib')
                    preds.append(joblib.load(pred_path))
                preds = np.array(preds)

                idx = np.where(tte_mask)[0]
                mae_vals = np.mean((preds[:, idx] * 40.76 - orig_y_test[idx] * 40.76), axis=1)
                mae_vals_by_model[model] = mae_vals

            # Print raw MAE results
            for model, name in zip(models, model_names):
                mae = np.median(mae_vals_by_model[model])
                lo = np.percentile(mae_vals_by_model[model], 2.5)
                hi = np.percentile(mae_vals_by_model[model], 97.5)
                print(f"{name}, {mae:.2f}, 95\\% CI: [{lo:.2f}, {hi:.2f}]")

            # Resampling test: Proposed better than each other model
            raw_pvals = []
            compared_models = []

            # for model in models:
            #     if model == 'proposed':
            #         continue
            #     diffs = mae_vals_by_model['proposed'] - mae_vals_by_model[model]
            #     pval = np.mean(diffs < 0)  # Proposed MAE < Other MAE
            #     raw_pvals.append(pval)
            #     compared_models.append(model)

            # # BH-FDR correction
            # reject, pvals_corrected, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')

            # print("\nOne-sided p-values (Proposed MAE < Other MAE):")
            # for model, p_raw, p_corr, rej in zip(compared_models, raw_pvals, pvals_corrected, reject):
            #     method_name = model_names[models.index(model)]
            #     star = "*" if rej else ""
            #     print(f"Proposed < {method_name}: p = {p_raw:.3f}, BH-adjusted p = {p_corr:.3f} {star}")
            # print()


==== covariate_high_resource ====
Censored — TTE <range days
IPCW, 576.64, 95\% CI: [463.50, 710.96]
Standard
Margin, 802.12, 95\% CI: [57.60, 1322.90]
Standard, 1326.73, 95\% CI: [1222.49, 1425.56]
Proposed - Linear, Small Weight, 271.99, 95\% CI: [234.80, 308.55]
Proposed - Exponential, Small Weight, 511.22, 95\% CI: [266.86, 758.05]
Proposed - Linear, Large Weight, 265.99, 95\% CI: [230.39, 301.68]
Proposed - Exponential, Large Weight, 581.65, 95\% CI: [314.45, 848.78]
Censored — TTE ≥range days
IPCW, -146.21, 95\% CI: [-246.14, -60.07]
Standard
Margin, -227.23, 95\% CI: [-1016.89, 284.07]
Standard, 389.53, 95\% CI: [334.24, 437.42]
Proposed - Linear, Small Weight, -514.19, 95\% CI: [-557.01, -467.74]
Proposed - Exponential, Small Weight, -88.30, 95\% CI: [-283.89, 6.15]
Proposed - Linear, Large Weight, -529.45, 95\% CI: [-572.85, -486.14]
Proposed - Exponential, Large Weight, -73.53, 95\% CI: [-296.60, 40.99]
Uncensored — TTE <range days
IPCW, 278.75, 95\% CI: [178.79, 429.99]
Stan

In [5]:
rows = []  # collect results here

for suffix in suffixes:
    orig_y_test = joblib.load(os.path.join(result_dir, f'orig_y_test_{suffix}.joblib'))
    binary_y_test = joblib.load(os.path.join(result_dir, f'binary_y_test_{suffix}.joblib'))

    masks = {
        'Censored': binary_y_test == 0,
        'Uncensored': binary_y_test == 1
    }

    for group_short, mask in masks.items():
        if not np.any(mask):
            continue

        tte_splits = {
            '<range days': mask & (orig_y_test < tte_threshold),
            '≥range days': mask & (orig_y_test >= tte_threshold)
        }

        for tte_label, tte_mask in tte_splits.items():
            if not np.any(tte_mask):
                continue

            # Collect MAE for each model
            mae_vals_by_model = {}
            for model in models:
                preds = []
                for splitidx in range(1000):
                    pred_path = os.path.join(preds_dir, f'{model}_y_test_pred_split{splitidx}_{suffix}.joblib')
                    preds.append(joblib.load(pred_path))
                preds = np.array(preds)

                idx = np.where(tte_mask)[0]
                # Compute MAE in days
                mae_vals = np.mean(np.abs(preds[:, idx] * 40.76 - orig_y_test[idx] * 40.76), axis=1)
                mae_vals_by_model[model] = mae_vals

            # Add rows for each model
            for model, name in zip(models, model_names):
                mae = np.median(mae_vals_by_model[model])
                lo = np.percentile(mae_vals_by_model[model], 2.5)
                hi = np.percentile(mae_vals_by_model[model], 97.5)
                rows.append({
                    'Group': group_short,
                    'TTE Range': tte_label,
                    'Model': name,
                    'Median MAE (days)': f"{mae:.2f}",
                    '95% CI': f"[{lo:.2f}, {hi:.2f}]"
                })

# Create DataFrame and show table
results_df = pd.DataFrame(rows)
print(results_df)


# Optional: save to CSV

         Group    TTE Range                                 Model  \
0     Censored  <range days                                  IPCW   
1     Censored  <range days                      Standard\nMargin   
2     Censored  <range days                              Standard   
3     Censored  <range days       Proposed - Linear, Small Weight   
4     Censored  <range days  Proposed - Exponential, Small Weight   
5     Censored  <range days       Proposed - Linear, Large Weight   
6     Censored  <range days  Proposed - Exponential, Large Weight   
7     Censored  ≥range days                                  IPCW   
8     Censored  ≥range days                      Standard\nMargin   
9     Censored  ≥range days                              Standard   
10    Censored  ≥range days       Proposed - Linear, Small Weight   
11    Censored  ≥range days  Proposed - Exponential, Small Weight   
12    Censored  ≥range days       Proposed - Linear, Large Weight   
13    Censored  ≥range days  Propo

In [6]:
import pandas as pd

df = results_df

# Separate data for <range and ≥range
ltrange = df[df["TTE Range"] == "<range days"]
gerange = df[df["TTE Range"] == "≥range days"]

# Helper: combine median and CI
def format_entry(median, ci):
    if pd.isna(median) or pd.isna(ci):
        return ""
    return f"{median} [{ci}]"

# Pivot median and CI separately
ltrange_median = ltrange.pivot(index="Model", columns="Group", values="Median MAE (days)")
ltrange_ci = ltrange.pivot(index="Model", columns="Group", values="95% CI")
gerange_median = gerange.pivot(index="Model", columns="Group", values="Median MAE (days)")
gerange_ci = gerange.pivot(index="Model", columns="Group", values="95% CI")

# Ensure all columns exist
for table in [ltrange_median, ltrange_ci, gerange_median, gerange_ci]:
    for col in ["Censored", "Uncensored"]:
        if col not in table.columns:
            table[col] = ""

# Start LaTeX lines
lines = []
lines.append("\\begin{table}[ht]")
lines.append("\\centering")
lines.append("\\scriptsize")
lines.append("\\begin{tabular}{lcccc}")
lines.append("\\toprule")
lines.append("& \\multicolumn{2}{c}{TTE <range days} & \\multicolumn{2}{c}{TTE ≥range days} \\\\")
lines.append("Model & Censored & Uncensored & Censored & Uncensored \\\\")
lines.append("\\midrule")

all_models = sorted(set(df["Model"]))

for model in all_models:
    lt_c = format_entry(ltrange_median.loc[model, "Censored"], ltrange_ci.loc[model, "Censored"]) if model in ltrange_median.index else ""
    lt_u = format_entry(ltrange_median.loc[model, "Uncensored"], ltrange_ci.loc[model, "Uncensored"]) if model in ltrange_median.index else ""
    ge_c = format_entry(gerange_median.loc[model, "Censored"], gerange_ci.loc[model, "Censored"]) if model in gerange_median.index else ""
    ge_u = format_entry(gerange_median.loc[model, "Uncensored"], gerange_ci.loc[model, "Uncensored"]) if model in gerange_median.index else ""
    lines.append(f"{model} & {lt_c} & {lt_u} & {ge_c} & {ge_u} \\\\")

lines.append("\\bottomrule")
lines.append("\\end{tabular}")
lines.append("\\caption{Median MAE and 95\\% CI by model, stratified by TTE range and censoring.}")
lines.append("\\label{tab:mae_combined}")
lines.append("\\end{table}")

# Save to file
latex_combined = "\n".join(lines)
print(latex_combined)

with open("mae_results_combined.tex", "w") as f:
    f.write(latex_combined)


\begin{table}[ht]
\centering
\scriptsize
\begin{tabular}{lcccc}
\toprule
& \multicolumn{2}{c}{TTE <range days} & \multicolumn{2}{c}{TTE ≥range days} \\
Model & Censored & Uncensored & Censored & Uncensored \\
\midrule
IPCW & 639.88 [[543.41, 760.28]] & 335.28 [[255.08, 472.03]] & 394.82 [[352.42, 456.32]] & 350.54 [[291.99, 423.92]] \\
Proposed - Exponential, Large Weight & 667.63 [[484.32, 875.48]] & 932.22 [[591.61, 1112.45]] & 378.34 [[308.10, 537.22]] & 309.03 [[257.90, 403.26]] \\
Proposed - Exponential, Small Weight & 591.72 [[420.28, 795.09]] & 581.16 [[365.49, 884.92]] & 366.77 [[303.75, 509.25]] & 309.03 [[254.19, 384.64]] \\
Proposed - Linear, Large Weight & 366.31 [[338.85, 395.05]] & 231.41 [[178.82, 323.45]] & 549.90 [[512.34, 589.25]] & 577.31 [[509.13, 651.44]] \\
Proposed - Linear, Small Weight & 368.52 [[338.49, 399.05]] & 232.73 [[170.93, 318.22]] & 536.77 [[495.44, 574.41]] & 559.52 [[486.90, 636.62]] \\
Standard & 1328.66 [[1224.20, 1427.31]] & 482.55 [[374.73, 586.